In [ ]:
# Import libraries

import scanpy as sc
import numpy as np
import pandas as pd
import scanpy as sc
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

In [ ]:
# Read all files

adata1 = sc.read_h5ad("Dataset_1.h5ad")
adata2 = sc.read_h5ad("Dataset_2.h5ad")
adata3 = sc.read_h5ad("Dataset_3.h5ad")
adata4 = sc.read_h5ad("Dataset_4.h5ad")
adata5 = sc.read_h5ad("Dataset_5.h5ad")
adata6 = sc.read_h5ad("Dataset_6.h5ad")
adata7 = sc.read_h5ad("Dataset_7.h5ad")
adata8 = sc.read_h5ad("Dataset_8.h5ad")
adata9 = sc.read_h5ad("Dataset_9.h5ad")

In [ ]:
# 1. Define the exact gene list extracted from the Supplementary Figure axes
target_genes = [
    "SORL1", "ARHGAP15", "ELMO1",
    "ETV6", "FRMD4A", "MEF2A",
    "ANKRD44", "ARHGAP26", "CHST11",
    "FKBP5", "ITPR2", "MED13L",
    "SRGAP2", "SRGAP2B", "TANC2",
    "ARHGAP22", "FMNL2", "MEF2C",
    "NAV3", "NHSL1", "PTPRC",
    "SPTLC2", "ACER3", "AOAH",
    "ARHGAP24", "DOCK8", "IL6ST"
]

# 2. Define standard dictionary of pre-loaded AnnData objects (Datasets 1-9)
# Ensure your objects are named matching these keys before running
all_adatas = {
    'DS1': adata1, 'DS2': adata2, 'DS3': adata3,
    'DS4': adata4, 'DS5': adata5, 'DS6': adata6,
    'DS7': adata7, 'DS8': adata8, 'DS9': adata9
}

# 3. Define the cluster mapping for Homeostatic and GPNMB+ microglial states
# Format: 'Dataset_ID': ([Homeostatic cluster IDs], [GPNMB+ cluster IDs])
microglia_mapping = {
    'DS1': ([5, 6, 7, 10, 13], [11]),
    'DS2': ([3, 5, 11, 14, 18], [12]),
    'DS3': ([0, 3, 10, 11, 12], [9]),
    'DS4': ([0, 1, 7, 8, 9, 10, 13], [2, 3]),
    'DS5': ([3, 5, 7, 14], [11]),
    'DS6': ([0, 13], [1, 2, 19, 20]),
    'DS7': ([3, 8], [9, 11, 12]),
    'DS8': ([0, 1], [2]),
    'DS9': ([2, 4, 5], [6, 8])
}

In [ ]:
def calculate_donor_state_logfc(adata_dict, cluster_mapping, gene_list):
    """
    Calculates per-donor Log2FoldChange between Active and Quiet microglial states.
    Aggregates technical replicates (e.g., _Rep1, _Rep2 in DS7) to ensure strictly 
    one biological observation per individual donor and avoid pseudoreplication.
    
    Parameters:
    -----------
    adata_dict : dict
        Dictionary containing loaded Scanpy AnnData objects.
    cluster_mapping : dict
        Dictionary defining Quiet and Active cluster IDs per dataset.
    gene_list : list
        List of target gene strings to process.
        
    Returns:
    --------
    pd.DataFrame
        Long-format master table containing biological donor-specific log2FC values.
    """
    master_records = []
    cluster_col = 'leiden_res_2.0'
    donor_col = 'sample'
    
    for ds_id, adata in adata_dict.items():
        if ds_id not in cluster_mapping:
            continue
            
        if cluster_col not in adata.obs.columns:
            print(f"Warning: '{cluster_col}' missing in {ds_id}. Skipping dataset.")
            continue
            
        quiet_cls, active_cls = cluster_mapping[ds_id]
        quiet_str = [str(c) for c in quiet_cls]
        active_str = [str(c) for c in active_cls]
        
        # Standardize donor IDs: strip technical replicate suffixes for DS7
        donor_series = adata.obs[donor_col].astype(str).copy()
        if ds_id == 'DS7':
            # Converts 'SM96_Monkey_SM45_Rep1' -> 'SM96_Monkey_SM45'
            donor_series = donor_series.str.replace(r'_Rep\d+$', '', regex=True)
            
        unique_donors = donor_series.unique()
        obs_clusters = adata.obs[cluster_col].astype(str)
        
        for gene_name in gene_list:
            if gene_name not in adata.var_names:
                continue
                
            gene_expr = adata[:, gene_name].X
            if hasattr(gene_expr, 'toarray'):
                gene_expr = gene_expr.toarray().flatten()
            else:
                gene_expr = np.array(gene_expr).flatten()
                
            # Loop over merged biological donors
            for donor in unique_donors:
                donor_mask = donor_series == donor
                
                donor_expr = gene_expr[donor_mask]
                donor_clusters = obs_clusters[donor_mask]
                
                quiet_mask = donor_clusters.isin(quiet_str)
                active_mask = donor_clusters.isin(active_str)
                
                if quiet_mask.sum() == 0 or active_mask.sum() == 0:
                    continue
                    
                mean_quiet = np.mean(donor_expr[quiet_mask])
                mean_active = np.mean(donor_expr[active_mask])
                donor_logfc = mean_active - mean_quiet
                
                master_records.append({
                    'Gene': gene_name,
                    'Dataset': ds_id,
                    'Donor': donor, # Contains the unified biological donor ID
                    'Log2FC_Active_vs_Quiet': donor_logfc
                })
                
    return pd.DataFrame(master_records)

In [ ]:
# 4. Execute execution pipeline and save checkpoints
print("Starting per-donor microglial state activation analysis...")
final_states_df = calculate_donor_state_logfc(all_adatas, microglia_mapping, target_genes)

# Export structural dataset table to csv
final_states_df.to_csv("microglia_donor_states_log2fc.csv", index=False)
print(f"Pipeline executed successfully! Total computed observations: {len(final_states_df)}")

In [ ]:
final_states_df

In [ ]:
# 1. Standardize and unify prefixes for the biological donor SM45 within DS7
mask_ds7 = final_states_df['Dataset'] == 'DS7'
final_states_df.loc[mask_ds7, 'Donor'] = final_states_df.loc[mask_ds7, 'Donor'].str.replace(r'^SM9(6|7)_', 'SM96_', regex=True)

# 2. Group the data by gene, dataset, and unified donor to calculate the mean Log2FC
final_states_df = final_states_df.groupby(['Gene', 'Dataset', 'Donor'], as_index=False)['Log2FC_Active_vs_Quiet'].mean()

# 3. Overwrite the final consolidated table into a CSV file
final_states_df.to_csv("microglia_donor_states_log2fc.csv", index=False)
print(f"Technical replicates merged successfully! New total observations: {len(final_states_df)}")

In [ ]:
# 1. Initialize lists to store statistical outputs per gene
lmm_records = []
unique_genes = final_states_df['Gene'].unique()

print(f"Running Linear Mixed-Effects Models (LMM) for {len(unique_genes)} genes...")

# 2. Iterate through each gene and fit a separate LMM
for gene in unique_genes:
    gene_df = final_states_df[final_states_df['Gene'] == gene]
    
    # Skip if dataset diversity is insufficient for random effect modeling
    if len(gene_df['Dataset'].unique()) < 2:
        print(f"Skipping {gene}: present in less than 2 datasets.")
        continue
        
    try:
        # Fit model estimating global baseline intercept with Dataset as a random intercept
        model = smf.mixedlm("Log2FC_Active_vs_Quiet ~ 1", data=gene_df, groups=gene_df["Dataset"])
        # Added method='bfgs' to improve optimization stability and reduce convergence warnings
        result = model.fit(method='bfgs')
        
        # Extract fixed-effect coefficients (Intercept represents the overall mean shift)
        intercept_coef = result.params['Intercept']
        p_value = result.pvalues['Intercept']
        
        lmm_records.append({
            'Gene': gene,
            'Intercept_Beta': intercept_coef,
            'P_Value': p_value,
            'Total_Donors': len(gene_df)
        })
    except Exception as e:
        print(f"Model fitting failed for gene {gene}: {str(e)}")

# 3. Compile statistics and apply FDR (Benjamini-Hochberg) multiple testing correction
lmm_results_df = pd.DataFrame(lmm_records)
lmm_results_df['FDR_Adjusted_P_Value'] = multipletests(lmm_results_df['P_Value'], method='fdr_bh')[1]

# 4. Classify functional gene status based on the direction and significance of the shift
def classify_gene_status(row, p_thresh=0.05):
    if row['FDR_Adjusted_P_Value'] >= p_thresh:
        return 'Neutral'
    return 'Activated' if row['Intercept_Beta'] > 0 else 'Homeostatic'

lmm_results_df['Functional_Status'] = lmm_results_df.apply(classify_gene_status, axis=1)

# 5. Sort final summary table by statistical significance and save
lmm_results_df = lmm_results_df.sort_values(by='FDR_Adjusted_P_Value', ascending=True).reset_index(drop=True)
lmm_results_df.to_csv("lmm_microglia_states_summary.csv", index=False)

In [ ]:
lmm_results_df

In [ ]:
def plot_single_gene_activation(gene_name, states_df, lmm_df):
    
    """
    Plots a single horizontal boxplot overlaying donor-level points.
    The box is rendered ON TOP of the points with alpha transparency to prevent blocking.
    """
    # 1. Filter data
    gene_data = states_df[states_df['Gene'] == gene_name].copy()
    lmm_sub = lmm_df[lmm_df['Gene'] == gene_name]
    
    if gene_data.empty or lmm_sub.empty:
        print(f"Warning: Data for gene {gene_name} not found. Skipping.")
        return
        
    mean_val = lmm_sub['Intercept_Beta'].values[0]
    raw_p = lmm_sub['FDR_Adjusted_P_Value'].values[0]
    p_str = f"FDR-Adjusted p-value = {raw_p:.3e}"
    
    ds_mapping = {
        'DS1': 'DS1', 'DS2': 'DS2', 'DS3_AC': 'DS3 (A)', 'DS3_FC': 'DS3 (F)',
        'DS4': 'DS4', 'DS5': 'DS5', 'DS6': 'DS6', 'DS8': 'DS8', 'DS9': 'DS9'
    }
    
    dataset_markers = {
        'DS1': 'o', 'DS2': 's', 'DS3 (A)': '^', 'DS3 (F)': 'v',
        'DS4': 'D', 'DS5': 'p', 'DS6': '*', 'DS8': 'h', 'DS9': 'X'
    }
    
    # 2. Setup vibrant divergent color gradient (narrowed limits for maximum saturation)
    cmap = plt.cm.coolwarm
    norm = mcolors.Normalize(vmin=-0.8, vmax=0.8)
    box_color = cmap(norm(mean_val))
    
    # 3. Canvas geometry layout (slightly taller to prevent clipping)
    sns.set_style("white")
    fig, ax = plt.subplots(figsize=(8, 2.2), dpi=300)
    
    # 4. FIRST LAYER: Render the scatter points behind the boxplot (zorder=2)
    np.random.seed(42)
    for _, row in gene_data.iterrows():
        raw_ds = row['Dataset']
        clean_ds = ds_mapping.get(raw_ds, raw_ds)
        marker_shape = dataset_markers.get(clean_ds, 'o')
        
        val = row['Log2FC_Active_vs_Quiet']
        jitter_y = np.random.uniform(-0.18, 0.18) # Slightly bounded jitter
        
        ax.scatter(val, 0 + jitter_y, 
                   c='#CCCCCC', marker=marker_shape, s=100,
                   edgecolors='#444444', linewidth=0.6, zorder=2, alpha=0.9)
        
    # 5. SECOND LAYER: Render the boxplot ON TOP with alpha transparency (zorder=4)
    box_plot = ax.boxplot(
        gene_data['Log2FC_Active_vs_Quiet'], vert=False, positions=[0],
        widths=0.45, patch_artist=True, showfliers=False, zorder=4,
        boxprops=dict(facecolor=box_color, color='black', linewidth=1.2, alpha=0.7),
        medianprops=dict(color='black', linewidth=1.8, zorder=5),
        whiskerprops=dict(color='black', linewidth=1.0, zorder=4),
        capprops=dict(color='black', linewidth=1.0, zorder=4)
    )
    
    # 6. Fine-tune axes, padding, and labels
    ax.axvline(0, color='#333333', linewidth=1.5, linestyle='-', zorder=1)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-0.55, 0.55)
    
    ax.set_yticks([])
    ax.set_yticklabels([])
    
    ax.set_xlabel('Log2FC (Active vs Quiet)', fontsize=11, fontweight='bold')
    ax.tick_params(axis='x', labelsize=9)
    ax.xaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
    sns.despine(left=True, ax=ax)
    
    ax.text(0.02, 1.1, f"{gene_name}", transform=ax.transAxes,
            fontsize=14, fontweight='bold', fontstyle='italic', va='center', ha='left')
            
    ax.text(0.98, 1.1, p_str, transform=ax.transAxes,
            fontsize=10, fontweight='bold', color='#333333', va='center', ha='right')
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_single_gene_activation(gene_name="SORL1", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="ARHGAP15", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="ELMO1", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="ETV6", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="FRMD4A", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="MEF2A", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="ANKRD44", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="ARHGAP26", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="CHST11", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="FKBP5", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="ITPR2", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="MED13L", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="SRGAP2", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="SRGAP2B", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="TANC2", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="ARHGAP22", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="FMNL2", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="MEF2C", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="NAV3", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="NHSL1", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="PTPRC", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="SPTLC2", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="ACER3", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="AOAH", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="ARHGAP24", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="DOCK8", states_df=final_states_df, lmm_df=lmm_results_df)

In [ ]:
plot_single_gene_activation(gene_name="IL6ST", states_df=final_states_df, lmm_df=lmm_results_df)